# 🔬 Analisi Statistica e Quantitativa: Pipeline VLM vs Ground Truth Temporale

**Autore**: Senior Python Developer & Data Analyst (Computer Vision & VLM Specialist)  
**Obiettivo**: Valutazione statistica avanzata e benchmark quantitativo confrontando le predizioni di modelli Vision-Language Models (VLM) con il Ground Truth (GT) valutato da esperti umani.  
**Output**: File Excel `risultati_analisi_metriche.xlsx` (9 fogli formattati professionalmente con dettaglio calcoli) e grafici temporali ad alta risoluzione in `output_plots/`.

---
## 📋 Architettura dell'Analisi
1. **Dinamicità Totale**: Adattamento automatico a qualsiasi video, durata o numero di finestre ($N$) tramite join su `window_id` (nessun valore hardcoded).
2. **8 Combinazioni Valutate**:
   - GT vs Run 1 (VLM Libero e Guidato)
   - GT vs Run 2 (VLM Libero e Guidato)
   - GT vs Run 3 (VLM Libero e Guidato)
   - GT vs Run Media (VLM Libero e Guidato)
3. **Metriche di Accuratezza su Due Livelli**:
   - *Modalità Complessiva*: Riferimento $GT_{\text{score}, t}$ reale su tutte le $N$ finestre.
   - *Modalità Solo Fuori Banda*: Riferimento limite di banda più vicino ($lower\_band_t$ o $upper\_band_t$) per le finestre fuori banda.
   - *Metriche*: Residui ($e_t$), Bias (Signed Mean Error), MAD/MAE, MSE, RMSE, MAPE (con $\epsilon = 10^{-6}$), Max Error.
4. **Metriche Intra-Run**: Valutazione della divergenza $\Delta_{\text{intra}}(t) = |y_{\text{libero}, t} - y_{\text{guidato}, t}|$ e $\text{MAD}_{\text{intra}}$.
5. **Metriche Inter-Run**: Variabilità tra Run 1, 2, 3 per finestra ($\bar{y}_t$, $d_{r,t}$, $\text{MAD}_{\text{inter}}$, $\sigma_t$, $CV_t\%$).
6. **Classificazione Livello di Rischio**: Matrici di Confusione, Accuracy complessiva e Recall per classe (`verde`, `giallo`, `arancione`, `rosso`).
7. **Workbook Excel Avanzato**: 9 fogli tematici formattati con `openpyxl` (stili, freeze panes, autofiltro, autowidth, formattazione condizionale verde/rosso e pagine con dettaglio calcoli).
8. **Visualizzazioni Grafiche**: 6 grafici ad alta risoluzione (150 DPI) per il profilo temporale, residui, variabilità e correlazioni.

## 1. Configurazione e Importazione Librerie

In [ ]:
# =============================================================================
# 1. IMPORTAZIONE LIBRERIE E SETUP GLOBALE
# =============================================================================
import os
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

# Calcolo numerico e manipolazione dataset
import numpy as np
import pandas as pd

# Grafica e visualizzazione avanzata
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

# Manipolazione e formattazione fogli Excel
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Soppressione warning per output pulito
warnings.filterwarnings("ignore")

# Opzioni di visualizzazione Pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 25)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# Configurazione stile grafici Matplotlib
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["axes.edgecolor"] = "#CCCCCC"
plt.rcParams["axes.linewidth"] = 0.8

# Costanti di analisi
EPSILON: float = 1e-6  # Protezione contro divisioni per zero nel MAPE
RISK_CLASSES: List[str] = ["verde", "giallo", "arancione", "rosso"]

# =============================================================================
# CONFIGURAZIONE FINESTRE DA ESCLUDERE DALLE METRICHE
# =============================================================================
# Inserire qui i Window ID che NON devono essere considerati nel calcolo
# delle metriche successive.
#
# Esempi:
#   EXCLUDED_WINDOWS = []           -> nessuna finestra esclusa
#   EXCLUDED_WINDOWS = [17]         -> esclude la finestra 17
#   EXCLUDED_WINDOWS = [3, 8, 17]   -> esclude le finestre 3, 8 e 17
#
# La configurazione viene applicata dopo il merge dei dati e prima
# dell'esecuzione di tutte le metriche successive.
EXCLUDED_WINDOWS = [17]


print("✅ Ambiente configurato correttamente.")
print(f"   Python: {os.sys.version.split()[0]} | Pandas: {pd.__version__} | OpenPyXL: {openpyxl.__version__}")
print(f"   Data e ora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Percorsi File di Input e Output

> ⚙️ **Configurazione Dinamica**: Modificare i percorsi seguenti in base ai file da analizzare. Il notebook gestirà dinamicamente qualsiasi dimensione $N$ di finestre temporali.

In [ ]:
# =============================================================================
# 2. DEFINIZIONE PERCORSI FILE (MODIFICABILI PER NUOVI VIDEO)
# =============================================================================

# ---- File Ground Truth (GT) ----
GT_FILE_PATH = Path(r"pipeline_scoring_output X Muletto incidente magazzino.xlsx")

# Nomi dei fogli supportati per il GT (con fallback automatico)
GT_PRIMARY_SHEET   = "pipeline_totale"
GT_SECONDARY_SHEET = "media_frame"

# Mappatura colonne del GT
GT_COLUMNS_MAP = {
    "window_id":      "window_id",
    "pipeline_score": "pipeline_score",
    "lower_band":     "lower_band",
    "upper_band":     "upper_band",
    "nearest_color":  "nearest_color",
    "start_time_s":   "start_time_s",
    "end_time_s":     "end_time_s",
}

# ---- File Esecuzioni VLM (3 Run distinte) ----
VLM_FILE_PATHS = [
    Path(r"pipeline_results_vlm_only X Muletto incidente magazzino 1.xlsx"),  # Run 1
    Path(r"pipeline_results_vlm_only X Muletto incidente magazzino 2.xlsx"),  # Run 2
    Path(r"pipeline_results_vlm_only X Muletto incidente magazzino 3.xlsx"),  # Run 3
]

# Nome del foglio nei file di output VLM
VLM_SHEET_NAME = "vlm_only_results"

# Nomi colonne presenti nel foglio VLM
VLM_COL_FREE_SCORE   = "dynamic_risk_score_vlm_free"
VLM_COL_FREE_CLASS   = "traffic_light_color_vlm_free"
VLM_COL_GUIDED_SCORE = "dynamic_risk_score_vlm"
VLM_COL_GUIDED_CLASS = "traffic_light_color_vlm"

# ---- File e Cartella di Output ----
OUTPUT_EXCEL_PATH = Path("risultati_analisi_metriche.xlsx")
OUTPUT_PLOTS_DIR  = Path("output_plots")
OUTPUT_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DPI = 150

print("📁 Configurazione percorsi completata.")
print(f"   • GT File:       {GT_FILE_PATH}")
for i, p in enumerate(VLM_FILE_PATHS, 1):
    print(f"   • VLM Run {i}:     {p}")
print(f"   • Output Excel:  {OUTPUT_EXCEL_PATH}")
print(f"   • Output Plots:  {OUTPUT_PLOTS_DIR}/")

## 3. Funzioni di Caricamento, Validazione e Mappatura

In [ ]:
# =============================================================================
# 3. FUNZIONI MODULARI DI CARICAMENTO E PARSING
# =============================================================================

def load_ground_truth(file_path: Path, primary_sheet: str, fallback_sheet: str, col_map: Dict[str, str]) -> pd.DataFrame:
    """
    Carica il file Ground Truth selezionando il foglio appropriato.
    Esegue la standardizzazione delle colonne e la conversione dei tipi.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File GT non trovato: {file_path.resolve()}")
    
    xl = pd.ExcelFile(file_path)
    available_sheets = xl.sheet_names
    
    if primary_sheet in available_sheets:
        chosen_sheet = primary_sheet
    elif fallback_sheet in available_sheets:
        chosen_sheet = fallback_sheet
        print(f"   ℹ️ Foglio '{primary_sheet}' non presente. Utilizzo fallback: '{fallback_sheet}'")
    else:
        chosen_sheet = available_sheets[0]
        print(f"   ⚠️ Utilizzo primo foglio disponibile: '{chosen_sheet}'")
    
    df = pd.read_excel(file_path, sheet_name=chosen_sheet)
    
    # Rinomina colonne secondo mapping
    rename_dict = {v: k for k, v in col_map.items() if v in df.columns and v != k}
    if rename_dict:
        df = df.rename(columns=rename_dict)
    
    req_cols = ["window_id", "pipeline_score", "lower_band", "upper_band", "nearest_color"]
    missing = [c for c in req_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colonne GT obbligatorie mancanti: {missing}. Presenti: {df.columns.tolist()}")
    
    # Pulizia e casting
    df = df.dropna(subset=["window_id"]).copy()
    df["window_id"] = df["window_id"].astype(int)
    df["pipeline_score"] = pd.to_numeric(df["pipeline_score"], errors="coerce")
    df["lower_band"]     = pd.to_numeric(df["lower_band"], errors="coerce")
    df["upper_band"]     = pd.to_numeric(df["upper_band"], errors="coerce")
    df["nearest_color"]  = df["nearest_color"].astype(str).str.lower().str.strip()
    
    keep_cols = [c for c in ["window_id", "start_time_s", "end_time_s", "pipeline_score", "lower_band", "upper_band", "nearest_color"] if c in df.columns]
    df = df[keep_cols].sort_values("window_id").reset_index(drop=True)
    print(f"✅ Ground Truth caricato ({chosen_sheet}): {len(df)} finestre temporali.")
    return df


def load_single_vlm_run(file_path: Path, sheet_name: str, run_id: int) -> pd.DataFrame:
    """
    Carica una singola run VLM e standardizza i nomi delle colonne con suffisso _r{run_id}.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File VLM Run {run_id} non trovato: {file_path.resolve()}")
    
    xl = pd.ExcelFile(file_path)
    chosen_sheet = sheet_name if sheet_name in xl.sheet_names else xl.sheet_names[0]
    df = pd.read_excel(file_path, sheet_name=chosen_sheet)
    
    expected = ["window_id", VLM_COL_FREE_SCORE, VLM_COL_FREE_CLASS, VLM_COL_GUIDED_SCORE, VLM_COL_GUIDED_CLASS]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Colonne mancanti in VLM Run {run_id}: {missing}")
    
    out = pd.DataFrame()
    out["window_id"] = df["window_id"].astype(int)
    out[f"score_free_r{run_id}"]   = pd.to_numeric(df[VLM_COL_FREE_SCORE], errors="coerce")
    out[f"class_free_r{run_id}"]   = df[VLM_COL_FREE_CLASS].astype(str).str.lower().str.strip()
    out[f"score_guided_r{run_id}"] = pd.to_numeric(df[VLM_COL_GUIDED_SCORE], errors="coerce")
    out[f"class_guided_r{run_id}"] = df[VLM_COL_GUIDED_CLASS].astype(str).str.lower().str.strip()
    
    out = out.sort_values("window_id").reset_index(drop=True)
    print(f"✅ VLM Run {run_id} caricata: {len(out)} finestre temporali.")
    return out


def score_to_traffic_light(score: float) -> str:
    """
    Mappatura standard punteggio continuo [0-100] -> classe di rischio:
    verde: [0, 25) | giallo: [25, 50) | arancione: [50, 75) | rosso: [75, 100]
    """
    if pd.isna(score):
        return "unknown"
    val = float(score)
    if val < 25.0:
        return "verde"
    elif val < 50.0:
        return "giallo"
    elif val < 75.0:
        return "arancione"
    else:
        return "rosso"

print("Funzioni di caricamento pronte.")

## 4. Merge Dinamico e Calcolo Run Media (8 Combinazioni di Confronto)

In questa sezione:
- Uniamo il Ground Truth e le 3 Run VLM tramite `inner join` su `window_id`.
- Rileviamo dinamicamente il numero di finestre $N$.
- Calcoliamo la **Run Media** (media per finestra $t$ tra Run 1, 2, 3) per VLM Libero e VLM Guidato.
- Definiamo le **8 combinazioni di confronto**:
  1. GT vs Run 1 – VLM libero – dynamic score
  2. GT vs Run 1 – VLM guidato – dynamic score
  3. GT vs Run 2 – VLM libero – dynamic score
  4. GT vs Run 2 – VLM guidato – dynamic score
  5. GT vs Run 3 – VLM libero – dynamic score
  6. GT vs Run 3 – VLM guidato – dynamic score
  7. GT vs Run Media – VLM libero – dynamic score
  8. GT vs Run Media – VLM guidato – dynamic score

In [ ]:
# =============================================================================
# 4. MERGE DATASET E CALCOLO RUN MEDIA
# =============================================================================

# Caricamento del Ground Truth
df_gt = load_ground_truth(GT_FILE_PATH, GT_PRIMARY_SHEET, GT_SECONDARY_SHEET, GT_COLUMNS_MAP)

# Caricamento delle 3 Run VLM
vlm_runs_dfs = [load_single_vlm_run(p, VLM_SHEET_NAME, i) for i, p in enumerate(VLM_FILE_PATHS, 1)]

# Merge progressivo su window_id
df_merged = df_gt.copy()
for i, df_v in enumerate(vlm_runs_dfs, 1):
    df_merged = pd.merge(df_merged, df_v, on="window_id", how="inner")

# Dimensione dinamica delle finestre N
N_WINDOWS = len(df_merged)
print(f"\n🔗 Merge completato: {len(df_merged)} finestre temporali individuate prima dell'eventuale esclusione.")

# Calcolo Run Media per VLM Libero e VLM Guidato
df_merged["score_free_mean"]   = df_merged[["score_free_r1", "score_free_r2", "score_free_r3"]].mean(axis=1)
df_merged["score_guided_mean"] = df_merged[["score_guided_r1", "score_guided_r2", "score_guided_r3"]].mean(axis=1)

# Assegnazione classe di rischio alla Run Media
df_merged["class_free_mean"]   = df_merged["score_free_mean"].apply(score_to_traffic_light)
df_merged["class_guided_mean"] = df_merged["score_guided_mean"].apply(score_to_traffic_light)

# =============================================================================
# ESCLUSIONE DELLE FINESTRE SELEZIONATE
# =============================================================================
# Da questo punto in poi, df_merged contiene esclusivamente le finestre
# ammesse al calcolo. Tutte le metriche successive (comparazioni, intra-run,
# inter-run, classificazione del rischio e relative visualizzazioni) useranno
# quindi automaticamente questo dataset filtrato.
#
# Il calcolo della Run Media sopra non viene alterato: la media è calcolata
# per ciascuna finestra tra Run 1, 2 e 3; le finestre indicate vengono poi
# escluse dall'analisi aggregata.
N_WINDOWS_TOTAL = len(df_merged)

# Normalizzazione e controllo degli ID inseriti dall'utente
EXCLUDED_WINDOWS = sorted(set(int(w) for w in EXCLUDED_WINDOWS))

available_window_ids = set(df_merged["window_id"].astype(int).tolist())
missing_excluded = sorted(set(EXCLUDED_WINDOWS) - available_window_ids)

if missing_excluded:
    print(
        f"   ⚠️ Attenzione: le seguenti finestre richieste per l'esclusione "
        f"non sono presenti nel dataset: {missing_excluded}"
    )

if EXCLUDED_WINDOWS:
    df_merged = df_merged[
        ~df_merged["window_id"].isin(EXCLUDED_WINDOWS)
    ].copy().reset_index(drop=True)

N_WINDOWS = len(df_merged)

print(
    f"🚫 Finestre escluse dal calcolo delle metriche: "
    f"{EXCLUDED_WINDOWS if EXCLUDED_WINDOWS else 'nessuna'}"
)
print(
    f"📊 Finestre disponibili prima dell'esclusione: {N_WINDOWS_TOTAL} | "
    f"finestre considerate nelle metriche: {N_WINDOWS}"
)

# Configurazione 8 combinazioni di confronto
COMPARISON_DEFS = [
    {"id": 1, "name": "GT vs Run 1 – VLM libero – dynamic score",  "run_name": "Run 1",     "branch": "VLM Libero",  "score_col": "score_free_r1",   "class_col": "class_free_r1"},
    {"id": 2, "name": "GT vs Run 1 – VLM guidato – dynamic score", "run_name": "Run 1",     "branch": "VLM Guidato", "score_col": "score_guided_r1", "class_col": "class_guided_r1"},
    {"id": 3, "name": "GT vs Run 2 – VLM libero – dynamic score",  "run_name": "Run 2",     "branch": "VLM Libero",  "score_col": "score_free_r2",   "class_col": "class_free_r2"},
    {"id": 4, "name": "GT vs Run 2 – VLM guidato – dynamic score", "run_name": "Run 2",     "branch": "VLM Guidato", "score_col": "score_guided_r2", "class_col": "class_guided_r2"},
    {"id": 5, "name": "GT vs Run 3 – VLM libero – dynamic score",  "run_name": "Run 3",     "branch": "VLM Libero",  "score_col": "score_free_r3",   "class_col": "class_free_r3"},
    {"id": 6, "name": "GT vs Run 3 – VLM guidato – dynamic score", "run_name": "Run 3",     "branch": "VLM Guidato", "score_col": "score_guided_r3", "class_col": "class_guided_r3"},
    {"id": 7, "name": "GT vs Run Media – VLM libero – dynamic score",  "run_name": "Run Media", "branch": "VLM Libero",  "score_col": "score_free_mean", "class_col": "class_free_mean"},
    {"id": 8, "name": "GT vs Run Media – VLM guidato – dynamic score", "run_name": "Run Media", "branch": "VLM Guidato", "score_col": "score_guided_mean","class_col": "class_guided_mean"},
]

print(f"\n📋 8 combinazioni di confronto configurate:")
for c in COMPARISON_DEFS:
    print(f"  [{c['id']}] {c['name']}")

df_merged.head(5)

## 5. Motore Matematico di Calcolo Metriche

Definizione delle metriche analitiche:
- **Condizione di Banda & Distanza ($d_t$)**:
  $$d_t = \begin{cases} 0 & \text{se } lower\_band_t \le y_t \le upper\_band_t \\ lower\_band_t - y_t & \text{se } y_t < lower\_band_t \\ y_t - upper\_band_t & \text{se } y_t > upper\_band_t \end{cases}$$
- **Metriche di Accuratezza**:
  - Residuo: $e_t = \hat{y}_t - y_{GT, t}$
  - Bias (Signed Mean Error): $\frac{1}{N} \sum e_t$
  - MAD (Mean Absolute Deviation): $\frac{1}{N} \sum |e_t|$
  - MSE: $\frac{1}{N} \sum e_t^2$
  - RMSE: $\sqrt{\text{MSE}}$
  - MAPE: $\frac{100\%}{N} \sum \left| \frac{e_t}{y_{GT, t} + \epsilon} \right|$

In [ ]:
# =============================================================================
# 5. MOTORE MATEMATICO DELLE METRICHE
# =============================================================================

def compute_distance_and_band(y_pred: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Calcola appartenenza alla banda e distanza dal limite.
    """
    inside_band = (y_pred >= lower) & (y_pred <= upper)
    distance = np.where(
        inside_band,
        0.0,
        np.where(y_pred < lower, lower - y_pred, y_pred - upper)
    )
    return inside_band, distance


def compute_statistical_metrics(y_pred: np.ndarray, y_ref: np.ndarray, eps: float = EPSILON) -> Dict[str, Any]:
    """
    Calcola il set completo di metriche statistiche.
    """
    mask = ~(np.isnan(y_pred) | np.isnan(y_ref))
    if mask.sum() == 0:
        return {
            "n": 0, "residuals": np.array([]), "bias": 0.0, "mad": 0.0,
            "mse": 0.0, "rmse": 0.0, "mape": 0.0, "max_ae": 0.0
        }
    
    yp = y_pred[mask]
    yr = y_ref[mask]
    n  = len(yp)
    
    residuals = yp - yr
    bias = np.mean(residuals)
    mad  = np.mean(np.abs(residuals))
    mse  = np.mean(residuals ** 2)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs(residuals) / (np.abs(yr) + eps)) * 100.0
    max_ae = np.max(np.abs(residuals))
    
    return {
        "n": n,
        "residuals": residuals,
        "bias": float(bias),
        "mad": float(mad),
        "mse": float(mse),
        "rmse": float(rmse),
        "mape": float(mape),
        "max_ae": float(max_ae)
    }


def evaluate_comparison(df: pd.DataFrame, comp: Dict[str, Any]) -> Dict[str, Any]:
    """
    Valuta una singola comparazione su tutti gli aspetti matematici richiesti.
    """
    y_pred    = df[comp["score_col"]].to_numpy(dtype=float)
    pred_cls  = df[comp["class_col"]].to_numpy(dtype=str)
    y_gt      = df["pipeline_score"].to_numpy(dtype=float)
    lower     = df["lower_band"].to_numpy(dtype=float)
    upper     = df["upper_band"].to_numpy(dtype=float)
    gt_cls    = df["nearest_color"].to_numpy(dtype=str)
    
    # 1. Banda e Distanza
    inside_band, distance = compute_distance_and_band(y_pred, lower, upper)
    n_total   = len(y_pred)
    n_inside  = int(inside_band.sum())
    n_outside = int((~inside_band).sum())
    agree_pct = (n_inside / n_total * 100.0) if n_total > 0 else 0.0
    
    dist_outside_vals = distance[distance > 0]
    mean_dist_outside = float(np.mean(dist_outside_vals)) if len(dist_outside_vals) > 0 else 0.0
    
    # Sovrastima / Sottostima
    n_over  = int((y_pred > upper).sum())
    n_under = int((y_pred < lower).sum())
    over_pct  = (n_over / n_total * 100.0) if n_total > 0 else 0.0
    under_pct = (n_under / n_total * 100.0) if n_total > 0 else 0.0
    
    # 2. Metriche Complessive (GT Reale)
    metrics_tot = compute_statistical_metrics(y_pred, y_gt)
    
    # 3. Metriche Solo Fuori Banda (GT Banda Limite più vicino)
    if n_outside > 0:
        out_idx = ~inside_band
        y_pred_out = y_pred[out_idx]
        lower_out  = lower[out_idx]
        upper_out  = upper[out_idx]
        y_gt_band_limit_out = np.where(y_pred_out < lower_out, lower_out, upper_out)
        metrics_out = compute_statistical_metrics(y_pred_out, y_gt_band_limit_out)
    else:
        metrics_out = {"n": 0, "bias": 0.0, "mad": 0.0, "mse": 0.0, "rmse": 0.0, "mape": 0.0, "max_ae": 0.0, "residuals": np.array([])}
    
    # 4. Concordanza Rischio
    risk_agree = (gt_cls == pred_cls)
    
    # 5. Dettaglio Finestra per Finestra (Foglio 2)
    df_win = pd.DataFrame({
        "comparison_name":       comp["name"],
        "window_id":             df["window_id"],
        "gt_score":              np.round(y_gt, 4),
        "gt_lower_band":         np.round(lower, 4),
        "gt_upper_band":         np.round(upper, 4),
        "run_score_evaluated":   np.round(y_pred, 4),
        "inside_gt_band":        inside_band,
        "distance_from_gt_band": np.round(distance, 4),
        "gt_risk_class":         gt_cls,
        "run_class_evaluated":   pred_cls,
        "risk_class_agreement":  risk_agree
    })
    
    # 6. Dettaglio Calcoli Complessivi (Foglio 3)
    res_tot = y_pred - y_gt
    df_calc_tot = pd.DataFrame({
        "comparison_name":       comp["name"],
        "window_id":             df["window_id"],
        "gt_score":              np.round(y_gt, 4),
        "model_score":           np.round(y_pred, 4),
        "residual_e":            np.round(res_tot, 4),
        "abs_residual":          np.round(np.abs(res_tot), 4),
        "squared_residual":      np.round(res_tot ** 2, 4),
        "percentage_abs_error":  np.round(np.abs(res_tot) / (np.abs(y_gt) + EPSILON) * 100.0, 4)
    })
    
    # 7. Dettaglio Calcoli Fuori Banda (Foglio 4)
    y_band_limit_all = np.where(inside_band, y_gt, np.where(y_pred < lower, lower, upper))
    res_band_all = y_pred - y_band_limit_all
    df_calc_out = pd.DataFrame({
        "comparison_name":       comp["name"],
        "window_id":             df["window_id"],
        "is_outside_band":       ~inside_band,
        "model_score":           np.round(y_pred, 4),
        "gt_band_limit":         np.round(y_band_limit_all, 4),
        "distance_residual":     np.round(np.where(~inside_band, res_band_all, 0.0), 4),
        "abs_distance_residual": np.round(np.where(~inside_band, np.abs(res_band_all), 0.0), 4),
        "squared_residual":      np.round(np.where(~inside_band, res_band_all ** 2, 0.0), 4),
        "percentage_abs_error":  np.round(np.where(~inside_band, np.abs(res_band_all) / (np.abs(y_band_limit_all) + EPSILON) * 100.0, 0.0), 4)
    })
    
    # 8. Riga per Foglio 1 (summary_metrics)
    summary_row = {
        "comparison":                 comp["name"],
        "source_type":                "Dynamic Score",
        "run_name":                   comp["run_name"],
        "branch":                     comp["branch"],
        "n_windows":                  n_total,
        "n_inside_gt_band":           n_inside,
        "n_outside_gt_band":          n_outside,
        "agreement_percentage":       round(agree_pct, 2),
        "mean_gt_score":              round(float(np.mean(y_gt)), 4),
        "mean_model_score":           round(float(np.mean(y_pred)), 4),
        "overestimation_percentage":   round(over_pct, 2),
        "underestimation_percentage":  round(under_pct, 2),
        "mae":                        round(metrics_tot["mad"], 4),
        "rmse":                       round(metrics_tot["rmse"], 4),
        "signed_mean_error":          round(metrics_tot["bias"], 4),
        "max_absolute_error":         round(metrics_tot["max_ae"], 4),
        "mean_distance_outside_band": round(mean_dist_outside, 4)
    }
    
    return {
        "comp": comp,
        "summary_row": summary_row,
        "df_win": df_win,
        "df_calc_tot": df_calc_tot,
        "df_calc_out": df_calc_out,
        "metrics_tot": metrics_tot,
        "metrics_out": metrics_out,
        "inside_band": inside_band,
        "distance": distance,
        "y_pred": y_pred,
        "y_gt": y_gt,
        "lower": lower,
        "upper": upper,
        "gt_cls": gt_cls,
        "pred_cls": pred_cls
    }

print("Motore di calcolo pronto.")

## 6. Esecuzione Analisi delle Comparazioni

In [ ]:
# =============================================================================
# 6. ESECUZIONE COMPARAZIONI E COSTRUZIONE TABELLE
# =============================================================================

results_list = [evaluate_comparison(df_merged, comp) for comp in COMPARISON_DEFS]

# Summary metrics ordinata decrescente su agreement_percentage
df_summary_metrics = pd.DataFrame([r["summary_row"] for r in results_list])
df_summary_metrics = df_summary_metrics.sort_values("agreement_percentage", ascending=False).reset_index(drop=True)

# Concatenazione dettagli finestre e calcoli
df_window_comparison = pd.concat([r["df_win"] for r in results_list], ignore_index=True)
df_calc_tot_all      = pd.concat([r["df_calc_tot"] for r in results_list], ignore_index=True)
df_calc_out_all      = pd.concat([r["df_calc_out"] for r in results_list], ignore_index=True)

# Tabelle di sintesi accuratezza
df_acc_tot_summary = pd.DataFrame([{
    "comparison":       r["comp"]["name"],
    "run_name":         r["comp"]["run_name"],
    "branch":           r["comp"]["branch"],
    "n_windows":        r["metrics_tot"]["n"],
    "bias_mean_error":  round(r["metrics_tot"]["bias"], 4),
    "mad_mae":          round(r["metrics_tot"]["mad"], 4),
    "mse":              round(r["metrics_tot"]["mse"], 4),
    "rmse":             round(r["metrics_tot"]["rmse"], 4),
    "mape_percentage":  round(r["metrics_tot"]["mape"], 4),
    "max_abs_error":    round(r["metrics_tot"]["max_ae"], 4)
} for r in results_list])

df_acc_out_summary = pd.DataFrame([{
    "comparison":          r["comp"]["name"],
    "run_name":            r["comp"]["run_name"],
    "branch":              r["comp"]["branch"],
    "n_outside_band":      r["summary_row"]["n_outside_gt_band"],
    "pct_outside":         round(r["summary_row"]["n_outside_gt_band"] / r["summary_row"]["n_windows"] * 100.0, 2),
    "bias_outside":        round(r["metrics_out"]["bias"], 4),
    "mad_mae_outside":     round(r["metrics_out"]["mad"], 4),
    "mse_outside":         round(r["metrics_out"]["mse"], 4),
    "rmse_outside":        round(r["metrics_out"]["rmse"], 4),
    "mape_pct_outside":    round(r["metrics_out"]["mape"], 4),
    "max_abs_err_outside": round(r["metrics_out"]["max_ae"], 4)
} for r in results_list])

print("📊 SUMMARY METRICS (Ordinato per agreement_percentage decrescente):")
df_summary_metrics[["comparison", "agreement_percentage", "mae", "rmse", "signed_mean_error", "mean_distance_outside_band"]]

## 7. Metriche Intra-Run (VLM Libero vs VLM Guidato)

In [ ]:
# =============================================================================
# 7. METRICHE INTRA-RUN
# =============================================================================

intra_configs = [
    ("Run 1",     "score_free_r1",   "score_guided_r1"),
    ("Run 2",     "score_free_r2",   "score_guided_r2"),
    ("Run 3",     "score_free_r3",   "score_guided_r3"),
    ("Run Media", "score_free_mean", "score_guided_mean"),
]

intra_summary_list = []
intra_detail_dfs   = []

for run_label, free_col, guided_col in intra_configs:
    y_f = df_merged[free_col].to_numpy(dtype=float)
    y_g = df_merged[guided_col].to_numpy(dtype=float)
    delta = np.abs(y_f - y_g)
    
    intra_summary_list.append({
        "run_name":      run_label,
        "mad_intra":     round(float(np.mean(delta)), 4),
        "min_delta":     round(float(np.min(delta)), 4),
        "max_delta":     round(float(np.max(delta)), 4),
        "median_delta":  round(float(np.median(delta)), 4),
        "std_delta":     round(float(np.std(delta, ddof=1)), 4),
        "n_windows":     len(delta)
    })
    
    intra_detail_dfs.append(pd.DataFrame({
        "run_name":        run_label,
        "window_id":       df_merged["window_id"],
        "score_free":      np.round(y_f, 4),
        "score_guided":    np.round(y_g, 4),
        "delta_intra_abs": np.round(delta, 4)
    }))

df_intra_summary = pd.DataFrame(intra_summary_list)
df_intra_detail  = pd.concat(intra_detail_dfs, ignore_index=True)

print("📌 SINTESI METRICHE INTRA-RUN:")
df_intra_summary

## 8. Metriche Inter-Run e Variabilità tra Run

In [ ]:
# =============================================================================
# 8. METRICHE INTER-RUN E VARIABILITÀ TRA RUN
# =============================================================================

def compute_inter_run_branch(df: pd.DataFrame, score_cols: List[str], branch_name: str) -> Dict[str, Any]:
    """
    Calcola variabilità e correlazioni per un ramo VLM includendo il Ground Truth (Matrice 5x5).
    """
    Y = df[score_cols].to_numpy(dtype=float)
    y_bar = np.mean(Y, axis=1)
    D = np.abs(Y - y_bar[:, np.newaxis])
    s_t = np.std(Y, axis=1, ddof=1)
    cv_t = (s_t / (np.abs(y_bar) + EPSILON)) * 100.0
    
    mad_inter_runs = np.mean(D, axis=0)
    summary_rows = []
    for i, r_name in enumerate(["Run 1", "Run 2", "Run 3"]):
        summary_rows.append({
            "branch":        branch_name,
            "run_name":      r_name,
            "mad_inter":     round(float(mad_inter_runs[i]), 4),
            "mean_std_dev":  round(float(np.mean(s_t)), 4),
            "mean_cv_pct":   round(float(np.mean(cv_t)), 2)
        })
    df_sum = pd.DataFrame(summary_rows)
    
    df_det = pd.DataFrame({
        "branch":           branch_name,
        "window_id":        df["window_id"],
        "score_run_1":      np.round(Y[:, 0], 4),
        "score_run_2":      np.round(Y[:, 1], 4),
        "score_run_3":      np.round(Y[:, 2], 4),
        "mean_score_y_bar": np.round(y_bar, 4),
        "dist_run_1":       np.round(D[:, 0], 4),
        "dist_run_2":       np.round(D[:, 1], 4),
        "dist_run_3":       np.round(D[:, 2], 4),
        "std_dev_s_t":      np.round(s_t, 4),
        "cv_percentage":    np.round(cv_t, 2)
    })
    
    
    return {
        "branch": branch_name,
        "df_summary": df_sum,
        "df_detail": df_det,
        "y_bar": y_bar,
        "s_t": s_t,
        "cv_t": cv_t
    }

inter_free   = compute_inter_run_branch(df_merged, ["score_free_r1", "score_free_r2", "score_free_r3"], "VLM Libero")
inter_guided = compute_inter_run_branch(df_merged, ["score_guided_r1", "score_guided_r2", "score_guided_r3"], "VLM Guidato")

df_inter_summary = pd.concat([inter_free["df_summary"], inter_guided["df_summary"]], ignore_index=True)
df_inter_detail  = pd.concat([inter_free["df_detail"], inter_guided["df_detail"]], ignore_index=True)

print("📌 SINTESI METRICHE INTER-RUN:")
print(df_inter_summary)

## 9. Classificazione del Livello di Rischio

In [ ]:
# =============================================================================
# 9. CLASSIFICAZIONE RISCHIO E MATRICI DI CONFUSIONE
# =============================================================================

risk_eval_list = []

for res in results_list:
    comp = res["comp"]
    gt_c = res["gt_cls"]
    pr_c = res["pred_cls"]
    
    conf_df = pd.DataFrame(0, index=RISK_CLASSES, columns=RISK_CLASSES)
    for g, p in zip(gt_c, pr_c):
        if g in RISK_CLASSES and p in RISK_CLASSES:
            conf_df.loc[g, p] += 1
    
    total_valid = conf_df.values.sum()
    correct     = np.trace(conf_df.values)
    acc         = (correct / total_valid * 100.0) if total_valid > 0 else 0.0
    
    recalls = {}
    for cls in RISK_CLASSES:
        gt_tot = conf_df.loc[cls].sum()
        recalls[cls] = round(conf_df.loc[cls, cls] / gt_tot * 100.0, 2) if gt_tot > 0 else None
    
    risk_eval_list.append({
        "comp_name":   comp["name"],
        "run_name":    comp["run_name"],
        "branch":      comp["branch"],
        "accuracy":    round(acc, 2),
        "correct":     int(correct),
        "total":       int(total_valid),
        "conf_matrix": conf_df,
        "recalls":     recalls
    })

df_risk_summary = pd.DataFrame([{
    "comparison":       r["comp_name"],
    "run_name":         r["run_name"],
    "branch":           r["branch"],
    "accuracy_pct":     r["accuracy"],
    "n_correct":        r["correct"],
    "n_total":          r["total"],
    "recall_verde_pct": r["recalls"].get("verde"),
    "recall_giallo_pct": r["recalls"].get("giallo"),
    "recall_arancione_pct": r["recalls"].get("arancione"),
    "recall_rosso_pct": r["recalls"].get("rosso")
} for r in risk_eval_list])

print("🚦 SINTESI ACCURACY CLASSI DI RISCHIO:")
df_risk_summary

## 10. Generazione File Excel di Output (`risultati_analisi_metriche.xlsx`)

In [ ]:
# =============================================================================
# 10. ESPORTAZIONE FILE EXCEL MULTI-FOGLIO CON OPENPYXL
# =============================================================================

wb = Workbook()
if "Sheet" in wb.sheetnames:
    del wb["Sheet"]

# Stili professionali
STYLE_HEADER_FILL    = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
STYLE_HEADER_FONT    = Font(color="FFFFFF", bold=True, name="Segoe UI", size=10)
STYLE_TITLE_FONT     = Font(color="1B365D", bold=True, name="Segoe UI", size=12)
STYLE_SUBTITLE_FONT  = Font(color="336699", bold=True, name="Segoe UI", size=10)
STYLE_DATA_FONT      = Font(name="Segoe UI", size=9)
STYLE_ALIGN_CENTER   = Alignment(horizontal="center", vertical="center")
STYLE_ALIGN_LEFT     = Alignment(horizontal="left", vertical="center")
STYLE_ALIGN_RIGHT    = Alignment(horizontal="right", vertical="center")

BORDER_THIN_SIDE     = Side(style="thin", color="D3D3D3")
STYLE_BORDER         = Border(left=BORDER_THIN_SIDE, right=BORDER_THIN_SIDE, top=BORDER_THIN_SIDE, bottom=BORDER_THIN_SIDE)

FILL_TRUE            = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
FONT_TRUE            = Font(color="006100", bold=True, name="Segoe UI", size=9)
FILL_FALSE           = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
FONT_FALSE           = Font(color="9C0006", bold=True, name="Segoe UI", size=9)

def style_worksheet(ws, header_row: int = 1, bool_cols: Optional[List[str]] = None) -> None:
    max_col = ws.max_column
    max_row = ws.max_row
    bool_cols = bool_cols or []
    
    col_names = {ws.cell(row=header_row, column=c).value: c for c in range(1, max_col + 1)}
    bool_col_idxs = [col_names[c] for c in bool_cols if c in col_names]
    
    for c in range(1, max_col + 1):
        cell = ws.cell(row=header_row, column=c)
        cell.fill = STYLE_HEADER_FILL
        cell.font = STYLE_HEADER_FONT
        cell.alignment = STYLE_ALIGN_CENTER
        cell.border = STYLE_BORDER
    
    for r in range(header_row + 1, max_row + 1):
        for c in range(1, max_col + 1):
            cell = ws.cell(row=r, column=c)
            cell.border = STYLE_BORDER
            cell.font = STYLE_DATA_FONT
            
            val = cell.value
            if c in bool_col_idxs:
                if val is True or str(val).lower() == "true":
                    cell.fill = FILL_TRUE
                    cell.font = FONT_TRUE
                elif val is False or str(val).lower() == "false":
                    cell.fill = FILL_FALSE
                    cell.font = FONT_FALSE
                cell.alignment = STYLE_ALIGN_CENTER
            else:
                if isinstance(val, (int, np.integer)):
                    cell.alignment = STYLE_ALIGN_RIGHT
                elif isinstance(val, (float, np.floating)):
                    cell.alignment = STYLE_ALIGN_RIGHT
                    cell.number_format = "0.0000"
                elif isinstance(val, str):
                    cell.alignment = STYLE_ALIGN_LEFT
                else:
                    cell.alignment = STYLE_ALIGN_CENTER
    
    ws.freeze_panes = f"A{header_row + 1}"
    ws.auto_filter.ref = f"A{header_row}:{get_column_letter(max_col)}{max_row}"
    
    for col in ws.columns:
        c_letter = get_column_letter(col[0].column)
        max_len = max([len(str(cell.value or '')) for cell in col if cell.row >= header_row] or [10])
        ws.column_dimensions[c_letter].width = min(max(max_len + 3, 10), 45)


def append_df(ws, df: pd.DataFrame, start_row: int = 1, start_col: int = 1) -> int:
    for col_idx, col_name in enumerate(df.columns, start=start_col):
        ws.cell(row=start_row, column=col_idx, value=col_name)
    
    for r_idx, row in enumerate(df.itertuples(index=False), start=start_row + 1):
        for c_idx, val in enumerate(row, start=start_col): precedence = 0
        for c_idx, val in enumerate(row, start=start_col):
            if isinstance(val, (np.integer,)):
                val = int(val)
            elif isinstance(val, (np.floating,)):
                val = float(val) if not np.isnan(val) else None
            elif isinstance(val, (np.bool_,)):
                val = bool(val)
            ws.cell(row=r_idx, column=c_idx, value=val)
    return start_row + len(df) + 1

# 1. summary_metrics
ws1 = wb.create_sheet(title="summary_metrics")
append_df(ws1, df_summary_metrics, start_row=1)
style_worksheet(ws1, header_row=1)

# 2. window_comparison
ws2 = wb.create_sheet(title="window_comparison")
append_df(ws2, df_window_comparison, start_row=1)
style_worksheet(ws2, header_row=1, bool_cols=["inside_gt_band", "risk_class_agreement"])

# 3. metriche_accuratezza_totale
ws3 = wb.create_sheet(title="metriche_accuratezza_totale")
ws3.cell(row=1, column=1, value="RIEPILOGO METRICHE ACCURATEZZA – GT REALE").font = STYLE_TITLE_FONT
next_r3 = append_df(ws3, df_acc_tot_summary, start_row=3)
ws3.cell(row=next_r3 + 1, column=1, value="DETTAGLIO CALCOLI RESIDUI PER FINESTRA TEMPORALE – GT REALE").font = STYLE_TITLE_FONT
append_df(ws3, df_calc_tot_all, start_row=next_r3 + 3)
style_worksheet(ws3, header_row=3)

# 4. metriche_accuratezza_fuori_banda
ws4 = wb.create_sheet(title="metriche_accuratezza_fuori_banda")
ws4.cell(row=1, column=1, value="RIEPILOGO METRICHE ACCURATEZZA – SOLO FUORI BANDA (GT LIMITE)").font = STYLE_TITLE_FONT
next_r4 = append_df(ws4, df_acc_out_summary, start_row=3)
ws4.cell(row=next_r4 + 1, column=1, value="DETTAGLIO CALCOLI PER FINESTRA – SOLO PUNTI FUORI BANDA").font = STYLE_TITLE_FONT
append_df(ws4, df_calc_out_all, start_row=next_r4 + 3)
style_worksheet(ws4, header_row=3, bool_cols=["is_outside_band"])

# 5. metriche_intra_run
ws5 = wb.create_sheet(title="metriche_intra_run")
ws5.cell(row=1, column=1, value="SINTESI METRICHE INTRA-RUN (|VLM Libero - VLM Guidato|)").font = STYLE_TITLE_FONT
next_r5 = append_df(ws5, df_intra_summary, start_row=3)
ws5.cell(row=next_r5 + 1, column=1, value="DETTAGLIO CALCOLI INTRA-RUN PER FINESTRA TEMPORALE").font = STYLE_TITLE_FONT
append_df(ws5, df_intra_detail, start_row=next_r5 + 3)
style_worksheet(ws5, header_row=3)

# 6. metriche_inter_run
ws6 = wb.create_sheet(title="metriche_inter_run")
ws6.cell(row=1, column=1, value="SINTESI METRICHE INTER-RUN (Variabilità Run 1, 2, 3)").font = STYLE_TITLE_FONT
next_r6 = append_df(ws6, df_inter_summary, start_row=3)
ws6.cell(row=next_r6 + 1, column=1, value="DETTAGLIO CALCOLI INTER-RUN (Media, Distanza, StdDev, CV%)").font = STYLE_TITLE_FONT
append_df(ws6, df_inter_detail, start_row=next_r6 + 3)
style_worksheet(ws6, header_row=3)

# 7. metriche_classi_rischio
ws8 = wb.create_sheet(title="metriche_classi_rischio")
ws8.cell(row=1, column=1, value="SINTESI ACCURACY CLASSIFICAZIONE RISCHIO").font = STYLE_TITLE_FONT
next_r8 = append_df(ws8, df_risk_summary, start_row=3)
curr_row_cm = next_r8 + 2
ws8.cell(row=curr_row_cm, column=1, value="MATRICI DI CONFUSIONE PER CIASCUNA COMPARAZIONE").font = STYLE_TITLE_FONT
curr_row_cm += 2

for r_eval in risk_eval_list:
    ws8.cell(row=curr_row_cm, column=1, value=f"{r_eval['comp_name']} | Accuracy: {r_eval['accuracy']}%").font = STYLE_SUBTITLE_FONT
    curr_row_cm += 1
    ws8.cell(row=curr_row_cm, column=1, value="GT / Predetto →").font = STYLE_HEADER_FONT
    ws8.cell(row=curr_row_cm, column=1).fill = STYLE_HEADER_FILL
    for col_i, cls_nm in enumerate(RISK_CLASSES, start=2):
        cell = ws8.cell(row=curr_row_cm, column=col_i, value=cls_nm)
        cell.font = STYLE_HEADER_FONT
        cell.fill = STYLE_HEADER_FILL
        cell.alignment = STYLE_ALIGN_CENTER
    
    cm = r_eval["conf_matrix"]
    for r_i, r_cls in enumerate(RISK_CLASSES, start=1):
        r_pos = curr_row_cm + r_i
        ws8.cell(row=r_pos, column=1, value=r_cls).font = Font(bold=True, name="Segoe UI", size=9)
        for c_i, c_cls in enumerate(RISK_CLASSES, start=2):
            val_cm = int(cm.loc[r_cls, c_cls])
            c_cell = ws8.cell(row=r_pos, column=c_i, value=val_cm)
            c_cell.alignment = STYLE_ALIGN_CENTER
            c_cell.border = STYLE_BORDER
            if r_cls == c_cls and val_cm > 0:
                c_cell.fill = FILL_TRUE
                c_cell.font = FONT_TRUE
            elif r_cls != c_cls and val_cm > 0:
                c_cell.fill = FILL_FALSE
                c_cell.font = FONT_FALSE
    curr_row_cm += len(RISK_CLASSES) + 3

style_worksheet(ws8, header_row=3)

# 8. riepilogo_dati_config
ws9 = wb.create_sheet(title="riepilogo_dati_config")
ws9.cell(row=1, column=1, value="METADATI E CONFIGURAZIONE ANALISI PIPELINE VLM").font = STYLE_TITLE_FONT
config_metadata = [
    ("Timestamp Esecuzione", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    ("File Ground Truth", str(GT_FILE_PATH)),
    ("Foglio Ground Truth", GT_PRIMARY_SHEET),
    ("Numero Finestre Considerate nelle Metriche (N)", int(N_WINDOWS)),
    ("Numero Finestre Disponibili Prima dell'Esclusione", int(N_WINDOWS_TOTAL)),
    ("Finestre Escluse dal Calcolo", ", ".join(map(str, EXCLUDED_WINDOWS)) if EXCLUDED_WINDOWS else "Nessuna"),
    ("Window ID Minimo", int(df_merged["window_id"].min())),
    ("Window ID Massimo", int(df_merged["window_id"].max())),
]
for i, p in enumerate(VLM_FILE_PATHS, 1):
    config_metadata.append((f"File VLM Run {i}", str(p)))
config_metadata.extend([
    ("Foglio VLM Results", VLM_SHEET_NAME),
    ("Epsilon MAPE", str(EPSILON)),
    ("File Excel Output", str(OUTPUT_EXCEL_PATH)),
    ("Cartella Output Grafici", str(OUTPUT_PLOTS_DIR)),
    ("DPI Grafici", int(PLOT_DPI))
])
if "end_time_s" in df_merged.columns:
    config_metadata.append(("Durata Totale Video (s)", float(df_merged["end_time_s"].max())))

df_cfg = pd.DataFrame(config_metadata, columns=["Parametro / Metadato", "Valore Impostato"])
append_df(ws9, df_cfg, start_row=3)
style_worksheet(ws9, header_row=3)

wb.save(OUTPUT_EXCEL_PATH)
print(f"\n💾 File Excel creato: {OUTPUT_EXCEL_PATH.resolve()}")
print(f"   Fogli ({len(wb.sheetnames)}): {wb.sheetnames}")

## 11. Visualizzazioni Grafiche ad Alta Risoluzione (`output_plots/`)

In [ ]:
# =============================================================================
# 11. GENERAZIONE GRAFICI AD ALTA RISOLUZIONE
# =============================================================================

def save_plot(fig, filename: str) -> None:
    p = OUTPUT_PLOTS_DIR / filename
    fig.savefig(p, dpi=PLOT_DPI, bbox_inches="tight", facecolor="white")
    print(f"    💾 Grafico salvato: {p}")

# GRAFICO 1: Profilo Temporale GT vs VLM
x_win = df_merged["window_id"].to_numpy()
y_gt_arr = df_merged["pipeline_score"].to_numpy()
low_arr  = df_merged["lower_band"].to_numpy()
upp_arr  = df_merged["upper_band"].to_numpy()

# Lasciamo spazio in basso per la legenda unica
fig1, axes1 = plt.subplots(
    2, 1,
    figsize=(16, 11),
    sharex=True,
    constrained_layout=False
)

fig1.suptitle(
    "Profilo Temporale di Rischio: Ground Truth vs Predizioni VLM",
    fontsize=14,
    fontweight="bold"
)

run_colors = {
    "r1": "#E74C3C",
    "r2": "#3498DB",
    "r3": "#2ECC71"
}

for ax_i, (b_name, col_prefix) in enumerate([
    ("VLM Libero", "score_free_"),
    ("VLM Guidato", "score_guided_")
]):
    ax = axes1[ax_i]

    # Aggiunta bande colorate orizzontali in background
    ax.axhspan(0, 25, color="#2ECC71", alpha=0.15, zorder=0)   # Verde
    ax.axhspan(25, 50, color="#F1C40F", alpha=0.15, zorder=0)  # Giallo
    ax.axhspan(50, 75, color="#E67E22", alpha=0.15, zorder=0)  # Arancione
    ax.axhspan(75, 105, color="#E74C3C", alpha=0.15, zorder=0) # Rosso

    ax.fill_between(
        x_win,
        low_arr,
        upp_arr,
        color="#BDC3C7",
        alpha=0.4,
        label="Fascia Confidenza GT",
        zorder=1
    )

    ax.plot(
        x_win,
        y_gt_arr,
        color="#2C3E50",
        linewidth=2.2,
        marker="o",
        markersize=4,
        label="GT Score (Esperto)",
        zorder=3
    )

    for r_i, r_lbl in enumerate(["r1", "r2", "r3"], 1):
        yp = df_merged[f"{col_prefix}{r_lbl}"].to_numpy()

        in_b, _ = compute_distance_and_band(
            yp,
            low_arr,
            upp_arr
        )

        ax.plot(
            x_win,
            yp,
            linestyle="--",
            color=run_colors[r_lbl],
            linewidth=1.2,
            alpha=0.85,
            label=f"Run {r_i}",
            zorder=4
        )

        ax.scatter(
            x_win[in_b],
            yp[in_b],
            color=run_colors[r_lbl],
            marker="o",
            s=40,
            edgecolors="white",
            linewidths=0.5,
            zorder=5
        )

        ax.scatter(
            x_win[~in_b],
            yp[~in_b],
            color=run_colors[r_lbl],
            marker="X",
            s=70,
            edgecolors="black",
            linewidths=0.5,
            zorder=6
        )

    yp_mean = df_merged[f"{col_prefix}mean"].to_numpy()

    ax.plot(
        x_win,
        yp_mean,
        color="#8E44AD",
        linewidth=2.5,
        linestyle="-",
        marker="s",
        markersize=5,
        label="Run Media",
        zorder=7
    )

    for th, col_th in [
        (25, "#F1C40F"),
        (50, "#E67E22"),
        (75, "#E74C3C")
    ]:
        ax.axhline(
            th,
            color=col_th,
            linestyle=":",
            alpha=0.6,
            linewidth=1
        )

    ax.set_title(
        b_name,
        fontsize=12,
        fontweight="bold"
    )

    ax.set_ylabel(
        "Dynamic Risk Score (0-100)",
        fontsize=10
    )

    ax.set_ylim(-5, 105)
    ax.set_xticks(x_win)

    ax.set_xticklabels(
        [str(int(w)) for w in x_win],
        rotation=45,
        fontsize=8
    )

    # Forza la visualizzazione dei tick sull'asse X anche sul primo grafico
    ax.tick_params(labelbottom=True)

    # NON inserire qui ax.legend()
    # La legenda verrà creata una sola volta sotto entrambi i grafici.

    ax.grid(True, alpha=0.3)


axes1[-1].set_xlabel(
    "Window ID (Finestra Temporale)",
    fontsize=10
)

# ============================================================================
# LEGENDA UNICA PER L'INTERA FIGURA
# ============================================================================

handles, labels = axes1[0].get_legend_handles_labels()

fig1.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=6,
    fontsize=9,
    frameon=True
)

# Spazio inferiore riservato alla legenda
fig1.subplots_adjust(
    top=0.92,
    bottom=0.12,
    hspace=0.25
)

save_plot(
    fig1,
    "grafico1_profilo_temporale_gt_vs_vlm.png"
)

plt.show()

In [ ]:
# GRAFICO 2: Residui Temporali per le Comparazioni
fig2, axes2 = plt.subplots(4, 2, figsize=(16, 18), constrained_layout=True)
fig2.suptitle("Residui Temporali (e_t = Model_Score - GT_Score) per le Comparazioni\n(Rosso = Sovrastima, Blu = Sottostima)", fontsize=14, fontweight="bold")

# Calcolo il massimo residuo assoluto globale per centrare a 0 e allineare tutti i grafici
max_abs_res = max([np.max(np.abs(res["y_pred"] - res["y_gt"])) for res in results_list])
y_lim_res = max_abs_res * 1.2 # Margine del 20%

for i, res in enumerate(results_list):
    ax = axes2[i // 2, i % 2]
    comp = res["comp"]
    res_vals = res["y_pred"] - res["y_gt"]
    bar_cols = ["#E74C3C" if r >= 0 else "#3498DB" for r in res_vals]
    
    ax.bar(x_win, res_vals, color=bar_cols, alpha=0.85, edgecolor="white", linewidth=0.4)
    ax.axhline(0, color="black", linewidth=1.2)
    
    bias_val = res["metrics_tot"]["bias"]
    ax.axhline(bias_val, color="#D35400", linestyle="--", linewidth=1.5, label=f"Bias = {bias_val:+.2f}")
    
    in_b = res["inside_band"]
    for w, r_v, ib in zip(x_win, res_vals, in_b):
        if not ib:
            offset = 1.5 if r_v >= 0 else -2.5
            ax.text(w, r_v + offset, "*", ha="center", va="center", fontsize=9, color="black", fontweight="bold")
    
    mae_v = res["summary_row"]["mae"]
    rmse_v = res["summary_row"]["rmse"]
    agr_v = res["summary_row"]["agreement_percentage"]
    ax.set_title(f"[{comp['id']}] {comp['name']}", fontsize=9, fontweight="bold")
    ax.text(0.02, 0.93, f"Accordo: {agr_v:.1f}% | MAE: {mae_v:.2f} | RMSE: {rmse_v:.2f}", transform=ax.transAxes, fontsize=8, bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.6))
    ax.set_ylabel("Residuo (e_t)", fontsize=8)
    
    # Applica i limiti simmetrici pre-calcolati
    ax.set_ylim(-y_lim_res, y_lim_res)
    
    ax.set_xticks(x_win)
    ax.set_xticklabels([str(int(w)) for w in x_win], rotation=45, fontsize=7)
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")

save_plot(fig2, "grafico2_residui_8_comparazioni.png")
plt.show()

# GRAFICO 3: Variabilità Inter-Run (CON ANNOTAZIONE * PER RUN MEDIA FUORI BANDA GT)
fig3, axes3 = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
fig3.suptitle("Variabilità Inter-Run per Finestra Temporale (Run 1, 2, 3)", fontsize=13, fontweight="bold")

# Calcolo massimi per asse Y per sigma e CV per avere ampiezze identiche
max_sigma = max(np.max(inter_free["s_t"]), np.max(inter_guided["s_t"]))
max_cv = max(np.max(inter_free["cv_t"]), np.max(inter_guided["cv_t"]))

for b_idx, (inter_data, b_label) in enumerate([(inter_free, "VLM Libero"), (inter_guided, "VLM Guidato")]):
    # Controllo superamento banda GT per la Run Media del rispettivo ramo
    mean_col = "score_free_mean" if b_label == "VLM Libero" else "score_guided_mean"
    y_mean_branch = df_merged[mean_col].to_numpy()
    out_of_band_mean = (y_mean_branch < low_arr) | (y_mean_branch > upp_arr)
    
    # --- Subplot Deviazione Standard (σ) ---
    ax_s = axes3[0, b_idx]
    s_vals = inter_data["s_t"]
    ax_s.bar(x_win, s_vals, color="#2980B9", alpha=0.85, edgecolor="white")
    ax_s.axhline(np.mean(s_vals), color="#C0392B", linestyle="--", linewidth=1.8, label=f"Media σ = {np.mean(s_vals):.2f}")
    ax_s.set_title(f"{b_label}: Deviazione Standard Inter-Run (σ_t)", fontsize=11, fontweight="bold")
    ax_s.set_ylabel("Deviazione Standard (σ)", fontsize=9)
    ax_s.set_ylim(0, max_sigma * 1.20)
    ax_s.set_xticks(x_win)
    ax_s.set_xticklabels([str(int(w)) for w in x_win], rotation=45, fontsize=7)
    ax_s.legend(fontsize=8)
    ax_s.grid(True, alpha=0.3, axis="y")
    
    # Aggiunta asterisco sopra la barra se la Run Media è fuori banda GT
    for w, s_v, is_out in zip(x_win, s_vals, out_of_band_mean):
        if is_out:
            ax_s.text(w, s_v + max_sigma * 0.02, "*", ha="center", va="bottom", fontsize=12, fontweight="bold", color="black")

    # --- Subplot Coefficiente di Variazione (CV%) ---
    ax_cv = axes3[1, b_idx]
    cv_vals = inter_data["cv_t"]
    cv_cols = ["#27AE60" if c < 15 else "#F39C12" if c < 30 else "#E74C3C" for c in cv_vals]
    ax_cv.bar(x_win, cv_vals, color=cv_cols, alpha=0.85, edgecolor="white")
    ax_cv.axhline(np.mean(cv_vals), color="#2C3E50", linestyle="--", linewidth=1.8, label=f"Media CV = {np.mean(cv_vals):.1f}%")
    ax_cv.axhline(15, color="#F39C12", linestyle=":", label="Soglia 15%")
    ax_cv.axhline(30, color="#E74C3C", linestyle=":", label="Soglia 30%")
    ax_cv.set_title(f"{b_label}: Coefficiente di Variazione (CV%)", fontsize=11, fontweight="bold")
    ax_cv.set_ylabel("CV (%)", fontsize=9)
    ax_cv.set_ylim(0, max_cv * 1.20)
    ax_cv.set_xlabel("Window ID", fontsize=9)
    ax_cv.set_xticks(x_win)
    ax_cv.set_xticklabels([str(int(w)) for w in x_win], rotation=45, fontsize=7)
    ax_cv.legend(fontsize=8)
    ax_cv.grid(True, alpha=0.3, axis="y")
    
    # Aggiunta asterisco sopra la barra se la Run Media è fuori banda GT
    for w, cv_v, is_out in zip(x_win, cv_vals, out_of_band_mean):
        if is_out:
            ax_cv.text(w, cv_v + max_cv * 0.02, "*", ha="center", va="bottom", fontsize=12, fontweight="bold", color="black")

save_plot(fig3, "grafico3_variabilita_inter_run.png")
plt.show()

# GRAFICO 4: Variabilità Intra-Run (CON SIMBOLI -, +, - + PER SUPERAMENTO BANDA GT)
fig4, axes4 = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
fig4.suptitle("Variabilità Intra-Run: |VLM Libero – VLM Guidato| per Finestra Temporale", fontsize=13, fontweight="bold")

# Calcolo la massima discrepanza per uniformare l'asse Y
max_intra = 0
for r_lbl, f_c, g_c in intra_configs:
    d = np.abs(df_merged[f_c].to_numpy(dtype=float) - df_merged[g_c].to_numpy(dtype=float))
    max_intra = max(max_intra, np.max(d))

for i, (r_lbl, f_c, g_c) in enumerate(intra_configs):
    ax = axes4[i // 2, i % 2]
    y_f = df_merged[f_c].to_numpy(dtype=float)
    y_g = df_merged[g_c].to_numpy(dtype=float)
    d_intra = np.abs(y_f - y_g)
    mad_val = float(np.mean(d_intra))
    
    b_cols = ["#27AE60" if d < 5 else "#F39C12" if d < 15 else "#E74C3C" for d in d_intra]
    ax.bar(x_win, d_intra, color=b_cols, alpha=0.85, edgecolor="white")
    ax.axhline(mad_val, color="#1B365D", linestyle="--", linewidth=1.8, label=f"MAD_intra = {mad_val:.2f}")
    
    # Verifica uscita dalla banda GT per ramo Libero e ramo Guidato
    free_out = (y_f < low_arr) | (y_f > upp_arr)
    guided_out = (y_g < low_arr) | (y_g > upp_arr)
    
    # Annotazione del simbolo sopra le barre
    for w, d_v, f_o, g_o in zip(x_win, d_intra, free_out, guided_out):
        if f_o and g_o:
            symbol = "- +"
        elif f_o:
            symbol = "-"
        elif g_o:
            symbol = "+"
        else:
            symbol = None
        
        if symbol:
            ax.text(w, d_v + max_intra * 0.02, symbol, ha="center", va="bottom", fontsize=10, fontweight="bold", color="black")

    ax.set_title(f"{r_lbl}: Discrepanza Assoluta Intra-Run", fontsize=11, fontweight="bold")
    ax.set_ylabel("|Libero - Guidato|", fontsize=9)
    ax.set_ylim(0, max_intra * 1.20)
    ax.set_xlabel("Window ID", fontsize=9)
    ax.set_xticks(x_win)
    ax.set_xticklabels([str(int(w)) for w in x_win], rotation=45, fontsize=7)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

save_plot(fig4, "grafico4_variabilita_intra_run.png")
plt.show()

# GRAFICO 5: Agreement % e MAE
fig6, axes6 = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig6.suptitle("Confronto Sinottico: Agreement % e MAE per le 8 Combinazioni", fontsize=13, fontweight="bold")

comp_names_short = [f"{r['summary_row']['run_name']}\n{r['comp']['branch'].replace('VLM ', '')}" for r in results_list]
agr_vals = [r["summary_row"]["agreement_percentage"] for r in results_list]
mae_vals = [r["summary_row"]["mae"] for r in results_list]
b_colors = ["#3498DB" if r["comp"]["branch"] == "VLM Libero" else "#E67E22" for r in results_list]

b1 = axes6[0].bar(range(len(comp_names_short)), agr_vals, color=b_colors, alpha=0.85, edgecolor="white")
axes6[0].set_title("Agreement % (Predizioni dentro Banda GT)", fontsize=11, fontweight="bold")
axes6[0].set_ylabel("Agreement (%)", fontsize=9)
axes6[0].set_xticks(range(len(comp_names_short)))
axes6[0].set_xticklabels(comp_names_short, fontsize=8)
axes6[0].set_ylim(0, 110)
axes6[0].grid(True, alpha=0.3, axis="y")
for bar, val in zip(b1, agr_vals):
    axes6[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f"{val:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")

b2 = axes6[1].bar(range(len(comp_names_short)), mae_vals, color=b_colors, alpha=0.85, edgecolor="white")
axes6[1].set_title("Mean Absolute Error (MAE)", fontsize=11, fontweight="bold")
axes6[1].set_ylabel("MAE", fontsize=9)
axes6[1].set_xticks(range(len(comp_names_short)))
axes6[1].set_xticklabels(comp_names_short, fontsize=8)
axes6[1].grid(True, alpha=0.3, axis="y")
for bar, val in zip(b2, mae_vals):
    axes6[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

patch_free = mpatches.Patch(color="#3498DB", label="VLM Libero")
patch_guid = mpatches.Patch(color="#E67E22", label="VLM Guidato")
fig6.legend(handles=[patch_free, patch_guid], loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05))

save_plot(fig6, "grafico5_riepilogo_agreement_mae.png")
plt.show()

## 12. Report Conclusivo

In [ ]:
# =============================================================================
# 12. REPORT FINALE DI SINTESI
# =============================================================================

print("=" * 80)
print("              REPORT CONCLUSIVO ANALISI PIPELINE VLM vs GROUND TRUTH")
print("=" * 80)
print(f"Dataset analizzato:     {N_WINDOWS} finestre temporali identificate dinamicamente.")
print(f"File Excel generato:    {OUTPUT_EXCEL_PATH.resolve()}")
print(f"Grafici PNG salvati in: {OUTPUT_PLOTS_DIR.resolve()}/")
print("-" * 80)
for _, row in df_summary_metrics.iterrows():
    print(f"{row['comparison']:<46} | Accordo: {row['agreement_percentage']:>5.1f}% | MAE: {row['mae']:>6.3f} | RMSE: {row['rmse']:>6.3f} | Bias: {row['signed_mean_error']:>+6.3f}")
print("=" * 80)

In [ ]:
# =============================================================================
# 13. HEATMAP MULTIDIMENSIONALE METRICHE ACCURATEZZA
# =============================================================================

# 1. Costruzione dataset esteso con metriche continue, categoriali e di trend
heatmap_data = []

for res in results_list:
    run_branch_label = (
        f"{res['comp']['run_name'].replace(' ', '')} - {res['comp']['branch']}"
    )

    # Estrazione accuratezza classificazione semaforica (punto 9 del notebook)
    risk_acc = df_risk_summary.loc[
        df_risk_summary["comparison"] == res["comp"]["name"], "accuracy_pct"
    ].values[0]

    heatmap_data.append({
        "comparison": run_branch_label,
        "agreement_percentage": res["summary_row"]["agreement_percentage"],
        "risk_accuracy": risk_acc,
        "mae": res["summary_row"]["mae"],
        "rmse": res["summary_row"]["rmse"],
        "mape": round(res["metrics_tot"]["mape"], 2),
        "signed_mean_error": res["summary_row"]["signed_mean_error"],
        "mean_distance_outside_band": res["summary_row"][
            "mean_distance_outside_band"
        ],
        "overestimation_percentage": res["summary_row"][
            "overestimation_percentage"
        ],
        "underestimation_percentage": res["summary_row"][
            "underestimation_percentage"
        ],
    })

df_heatmap = pd.DataFrame(heatmap_data).set_index("comparison")

# Ordinamento dalla migliore alla peggiore basato su agreement_percentage
df_heatmap = df_heatmap.sort_values(
    by="agreement_percentage",
    ascending=False
)

# Sostituzione dell'underscore '_' con lo spazio ' ' in tutte le colonne e indici
df_heatmap.columns = [
    col.replace("_", " ") for col in df_heatmap.columns
]

df_heatmap.index.name = df_heatmap.index.name.replace("_", " ")

# 2. Normalizzazione direzionale automatica
#    (1.0 = Migliore/Verde, 0.0 = Peggiore/Rosso)
higher_is_better = [
    "agreement percentage",
    "risk accuracy"
]

df_norm = pd.DataFrame(index=df_heatmap.index)

for col in df_heatmap.columns:
    min_v, max_v = df_heatmap[col].min(), df_heatmap[col].max()
    denom = (max_v - min_v) if max_v != min_v else 1.0

    if col in higher_is_better:
        df_norm[col] = (df_heatmap[col] - min_v) / denom
    else:
        df_norm[col] = (max_v - df_heatmap[col]) / denom

# 3. Generazione e salvataggio della Heatmap avanzata
fig_hm, ax_hm = plt.subplots(
    figsize=(18, 7),
    constrained_layout=True
)

sns.heatmap(
    df_norm,
    annot=df_heatmap,
    fmt="g",
    cmap="RdYlGn",
    linewidths=0.5,
    vmin=0,
    vmax=1,
    ax=ax_hm,
    cbar_kws={
        "label": "Performance Relativa (0 = Peggiore, 1 = Migliore)"
    },
)

ax_hm.set_title(
    "Heatmap Multidimensionale Metriche Accuratezza",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

ax_hm.set_ylabel(
    df_heatmap.index.name,
    fontsize=11,
    fontweight="bold"
)

ax_hm.set_xticklabels(
    ax_hm.get_xticklabels(),
    rotation=30,
    ha="right",
    fontsize=10
)

ax_hm.set_yticklabels(
    ax_hm.get_yticklabels(),
    rotation=0,
    fontsize=10
)

# Salvataggio nella cartella dei grafici configurata
save_plot(
    fig_hm,
    "grafico6_heatmap_metriche_accuratezza.png"
)

plt.show()

In [ ]:
# =============================================================================
# GRAFICO AGGIUNTIVO: QUANTITÀ FUORI BANDA TEMPORALE
# =============================================================================
#
# Struttura analoga al Grafico 2:
#   - 8 subplot (4x2), uno per comparazione
#   - X = Window ID
#   - Y = Quantità fuori banda
#   - Asterisco sulle finestre fuori banda
#   - Box informativo: % Fuori banda | MAE | RMSE
#   - Legenda in alto a destra
#
# La "quantità fuori banda" corrisponde alla distanza della predizione
# dal limite più vicino della banda GT.
# Se la predizione è dentro banda -> distanza = 0.
# =============================================================================

fig_outband, axes_outband = plt.subplots(
    4,
    2,
    figsize=(16, 18),
    constrained_layout=True
)

fig_outband.suptitle(
    "Quantità Fuori Banda per le Comparazioni\n"
    "(Distanza della predizione dal limite della Banda GT)",
    fontsize=14,
    fontweight="bold"
)

# Appiattimento degli assi per iterare facilmente sulle 8 comparazioni
axes_outband = axes_outband.flatten()


# -----------------------------------------------------------------------------
# Calcolo del massimo valore di distanza fuori banda globale
# per mantenere la stessa scala Y in tutti i subplot
# -----------------------------------------------------------------------------

max_distance_out = max(
    [
        np.max(res["distance"])
        if len(res["distance"]) > 0
        else 0.0
        for res in results_list
    ]
)

# Margine superiore del 20%
y_lim_outband = max_distance_out * 1.20

# Gestione del caso limite in cui tutte le predizioni siano dentro banda
if y_lim_outband == 0:
    y_lim_outband = 1.0


# -----------------------------------------------------------------------------
# Creazione degli 8 subplot
# -----------------------------------------------------------------------------

for i, res in enumerate(results_list):

    ax = axes_outband[i]

    comp = res["comp"]

    # Quantità fuori banda:
    # 0 = predizione dentro la banda GT
    # > 0 = distanza dal limite più vicino della banda GT
    distance_vals = res["distance"]

    # Identificazione delle finestre fuori banda
    in_b = res["inside_band"]
    out_b = ~in_b

    # -------------------------------------------------------------------------
    # Barre
    # -------------------------------------------------------------------------
    #
    # Tutti i valori sono >= 0 perché rappresentano una distanza.
    # Le barre fuori banda sono evidenziate.
    # -------------------------------------------------------------------------

    bar_cols = [
        "#E74C3C" if d > 0 else "#BDC3C7"
        for d in distance_vals
    ]

    ax.bar(
        x_win,
        distance_vals,
        color=bar_cols,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
        label="Quantità fuori banda"
    )

    # -------------------------------------------------------------------------
    # Linea dello zero
    # -------------------------------------------------------------------------

    ax.axhline(
        0,
        color="black",
        linewidth=1.2
    )


    # -------------------------------------------------------------------------
    # Asterisco sulle finestre fuori banda
    # -------------------------------------------------------------------------

    for w, d_v, is_out in zip(
        x_win,
        distance_vals,
        out_b
    ):

        if is_out:

            # Piccolo offset proporzionale all'asse Y
            offset = max(y_lim_outband * 0.025, 0.3)

            ax.text(
                w,
                d_v + offset,
                "*",
                ha="center",
                va="bottom",
                fontsize=9,
                color="black",
                fontweight="bold"
            )


    # -------------------------------------------------------------------------
    # Metriche aggregate relative alle sole osservazioni fuori banda
    # -------------------------------------------------------------------------

    n_total = len(distance_vals)
    n_outside = int(np.sum(out_b))

    # Percentuale di finestre fuori banda
    outside_pct = (
        n_outside / n_total * 100.0
        if n_total > 0
        else 0.0
    )

    # MAE e RMSE calcolati sulle sole osservazioni fuori banda.
    #
    # metrics_out confronta la predizione fuori banda con il limite
    # della banda GT più vicino.
    mae_out = res["metrics_out"]["mad"]
    rmse_out = res["metrics_out"]["rmse"]


    # -------------------------------------------------------------------------
    # Titolo del subplot
    # -------------------------------------------------------------------------

    ax.set_title(
        f"[{comp['id']}] {comp['name']}",
        fontsize=9,
        fontweight="bold"
    )


    # -------------------------------------------------------------------------
    # Box informativo
    # -------------------------------------------------------------------------

    ax.text(
        0.02,
        0.93,
        f"Fuori banda: {outside_pct:.1f}% | "
        f"MAE: {mae_out:.2f} | "
        f"RMSE: {rmse_out:.2f}",
        transform=ax.transAxes,
        fontsize=8,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="wheat",
            alpha=0.6
        )
    )


    # -------------------------------------------------------------------------
    # Etichette assi
    # -------------------------------------------------------------------------

    ax.set_ylabel(
        "Quantità fuori banda",
        fontsize=8
    )

    ax.set_xlabel(
        "Window ID",
        fontsize=8
    )


    # -------------------------------------------------------------------------
    # Limiti asse Y uniformi
    # -------------------------------------------------------------------------

    ax.set_ylim(
        0,
        y_lim_outband
    )


    # -------------------------------------------------------------------------
    # Asse X
    # -------------------------------------------------------------------------

    ax.set_xticks(x_win)

    ax.set_xticklabels(
        [str(int(w)) for w in x_win],
        rotation=45,
        fontsize=7
    )


    # -------------------------------------------------------------------------
    # Legenda
    #
    # Stessa posizione del Grafico 2: upper right
    # -------------------------------------------------------------------------

    ax.legend(
        loc="upper right",
        fontsize=7
    )


    # -------------------------------------------------------------------------
    # Griglia
    # -------------------------------------------------------------------------

    ax.grid(
        True,
        alpha=0.3,
        axis="y"
    )


# -----------------------------------------------------------------------------
# Salvataggio
# -----------------------------------------------------------------------------

save_plot(
    fig_outband,
    "grafico7_quantita_fuori_banda_8_comparazioni.png"
)

plt.show()

In [ ]:
# =============================================================================
# GRAFICO AGGIUNTIVO: RUN MEDIA - QUANTITÀ FUORI BANDA
# =============================================================================
#
# Visualizza esclusivamente le due Run Media:
#   [7] GT vs Run Media - VLM libero
#   [8] GT vs Run Media - VLM guidato
#
# X = Window ID
# Y = Quantità fuori banda
#
# La quantità fuori banda è la distanza della predizione dal limite della
# banda GT più vicino. Per i punti dentro banda il valore è 0.
#
# Box informativo:
#   Fuori banda (%) | MAE | RMSE
#
# Asterisco = finestra fuori banda
# =============================================================================


# -----------------------------------------------------------------------------
# Selezione esclusiva delle due comparazioni "Run Media"
# -----------------------------------------------------------------------------

run_media_results = [
    res for res in results_list
    if res["comp"]["run_name"] == "Run Media"
]


# Controllo di sicurezza
if len(run_media_results) != 2:
    raise ValueError(
        f"Attese 2 comparazioni Run Media, trovate {len(run_media_results)}."
    )


# -----------------------------------------------------------------------------
# Creazione figura
# -----------------------------------------------------------------------------

fig_mean_outband, axes_mean_outband = plt.subplots(
    1,
    2,
    figsize=(16, 6),
    constrained_layout=True
)

fig_mean_outband.suptitle(
    "Quantità Fuori Banda per le Run Media\n"
    "(VLM Libero vs VLM Guidato)",
    fontsize=14,
    fontweight="bold"
)


# -----------------------------------------------------------------------------
# Calcolo massimo globale della quantità fuori banda
# per mantenere la stessa scala Y nei due grafici
# -----------------------------------------------------------------------------

max_distance_mean = max(
    np.max(res["distance"])
    for res in run_media_results
    if len(res["distance"]) > 0
)

y_lim_mean_outband = max_distance_mean * 1.20

# Caso limite: nessun punto fuori banda
if y_lim_mean_outband == 0:
    y_lim_mean_outband = 1.0


# -----------------------------------------------------------------------------
# Creazione dei due subplot
# -----------------------------------------------------------------------------

for i, res in enumerate(run_media_results):

    ax = axes_mean_outband[i]

    comp = res["comp"]

    # Quantità fuori banda:
    # 0 -> dentro banda
    # > 0 -> distanza dal limite della banda GT
    distance_vals = res["distance"]

    # Identificazione punti dentro/fuori banda
    in_b = res["inside_band"]
    out_b = ~in_b


    # -------------------------------------------------------------------------
    # Colori delle barre
    # -------------------------------------------------------------------------
    #
    # Grigio = punto dentro banda (quantità fuori banda = 0)
    # Rosso  = punto fuori banda
    # -------------------------------------------------------------------------

    bar_cols = [
        "#E74C3C" if d > 0 else "#BDC3C7"
        for d in distance_vals
    ]


    ax.bar(
        x_win,
        distance_vals,
        color=bar_cols,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
        label="Quantità fuori banda"
    )


    # -------------------------------------------------------------------------
    # Linea dello zero
    # -------------------------------------------------------------------------

    ax.axhline(
        0,
        color="black",
        linewidth=1.2
    )


    # -------------------------------------------------------------------------
    # Asterisco sui punti fuori banda
    # -------------------------------------------------------------------------

    for w, d_v, is_out in zip(
        x_win,
        distance_vals,
        out_b
    ):

        if is_out:

            offset = max(y_lim_mean_outband * 0.025, 0.3)

            ax.text(
                w,
                d_v + offset,
                "*",
                ha="center",
                va="bottom",
                fontsize=9,
                color="black",
                fontweight="bold"
            )


    # -------------------------------------------------------------------------
    # Calcolo percentuale fuori banda
    # -------------------------------------------------------------------------

    n_total = len(distance_vals)
    n_outside = int(np.sum(out_b))

    pct_outside = (
        n_outside / n_total * 100.0
        if n_total > 0
        else 0.0
    )


    # -------------------------------------------------------------------------
    # Metriche calcolate esclusivamente sui punti fuori banda
    # -------------------------------------------------------------------------

    mae_out = res["metrics_out"]["mad"]
    rmse_out = res["metrics_out"]["rmse"]


    # -------------------------------------------------------------------------
    # Titolo
    # -------------------------------------------------------------------------

    ax.set_title(
        f"[{comp['id']}] {comp['name']}",
        fontsize=10,
        fontweight="bold"
    )


    # -------------------------------------------------------------------------
    # Box con le metriche
    # -------------------------------------------------------------------------

    ax.text(
        0.02,
        0.93,
        f"Fuori banda: {pct_outside:.1f}% | "
        f"MAE: {mae_out:.2f} | "
        f"RMSE: {rmse_out:.2f}",
        transform=ax.transAxes,
        fontsize=8,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="wheat",
            alpha=0.6
        )
    )


    # -------------------------------------------------------------------------
    # Etichette assi
    # -------------------------------------------------------------------------

    ax.set_ylabel(
        "Quantità fuori banda",
        fontsize=9
    )

    ax.set_xlabel(
        "Window ID",
        fontsize=9
    )


    # -------------------------------------------------------------------------
    # Scala Y comune
    # -------------------------------------------------------------------------

    ax.set_ylim(
        0,
        y_lim_mean_outband
    )


    # -------------------------------------------------------------------------
    # Asse X
    # -------------------------------------------------------------------------

    ax.set_xticks(x_win)

    ax.set_xticklabels(
        [str(int(w)) for w in x_win],
        rotation=45,
        fontsize=8
    )


    # -------------------------------------------------------------------------
    # Legenda
    # Stessa posizione utilizzata nel Grafico 2
    # -------------------------------------------------------------------------

    ax.legend(
        loc="upper right",
        fontsize=7
    )


    # -------------------------------------------------------------------------
    # Griglia
    # -------------------------------------------------------------------------

    ax.grid(
        True,
        alpha=0.3,
        axis="y"
    )


# -----------------------------------------------------------------------------
# Salvataggio
# -----------------------------------------------------------------------------

save_plot(
    fig_mean_outband,
    "grafico8_run_media_quantita_fuori_banda.png"
)

plt.show()